# Revisión de la hoja de rasgos (`Rasgos_2025`) — `Sumaco_17092025.xlsx`

Notebook corto para abrir la hoja de rasgos, ver los rangos de cada variable numérica y marcar valores que no cuadran (negativos, hoja seca más pesada que hoja fresca, texto donde debería haber un número, valores muy alejados del resto).

## 1. Cargar la hoja

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)

df = pd.read_excel('Sumaco_17092025.xlsx', sheet_name='Rasgos_2025')
print(df.shape)
df.head()

## 2. Rangos de las variables numéricas

`describe()` sobre las columnas de rasgos (DBH, altura, peso y grosor de hoja, rasgos de madera) para ver mínimo, máximo, media y cuartiles de un vistazo.

In [ ]:
rasgo_cols = ['dbh 2', 'tree height 2011 (m)', 'No. leaves',
              'Leaf fresh weight (g)', 'Leaf dry weight (g)',
              'leaf 1 thickness 1', 'leaf 1 thickness 2',
              'leaf 2 thickness 1', 'leaf 2 thickness 2',
              'leaf 3 thickness 1', 'leaf 3 thickness 2',
              'crust thickness', 'wet lenght cm', 'wet wood weight', 'dry wood weight']

df[rasgo_cols].describe().T

## 3. `dbh 1` viene con texto en vez de número

Algunas filas tienen un texto tipo `'tree (5cm≤dbh<10cm)'` en vez de un valor numérico de DBH (parece un rango puesto a mano cuando no se midió el árbol exacto). Se separa en `dbh_1_num` (numérico, NaN si era texto) y `dbh_1_texto` para no perder esa info.

In [ ]:
def a_numero(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return pd.NA

df['dbh_1_num'] = df['dbh 1'].apply(a_numero)
df['dbh_1_texto'] = df['dbh 1'].where(df['dbh_1_num'].isna() & df['dbh 1'].notna())

print('Filas con dbh 1 como texto:', df['dbh_1_texto'].notna().sum())
df[df['dbh_1_texto'].notna()][['plot ID', 'treeID', 'dbh 1']]

## 4. Chequeos de consistencia (QC)

- Peso seco de hoja mayor al peso fresco (imposible).
- Peso seco de madera mayor al peso húmedo (imposible).
- Grosores de hoja o pesos en cero o negativos.
- Valores muy alejados del resto (outliers): fuera de `mediana ± 3 × MAD` para cada columna numérica.

In [ ]:
df['QC_flag'] = ''

m = df['Leaf dry weight (g)'] > df['Leaf fresh weight (g)']
df.loc[m, 'QC_flag'] += 'peso seco de hoja > peso fresco; '

m = df['dry wood weight'] > df['wet wood weight']
df.loc[m, 'QC_flag'] += 'peso seco de madera > peso humedo; '

for c in rasgo_cols:
    m = df[c] <= 0
    df.loc[m, 'QC_flag'] += f'{c} <= 0; '

def marca_outliers(col):
    s = df[col].dropna()
    mediana = s.median()
    mad = (s - mediana).abs().median()
    if mad == 0:
        return
    limite = 3 * mad * 1.4826  # MAD escalado a std aproximada
    out = df[col].sub(mediana).abs() > limite
    df.loc[out.fillna(False), 'QC_flag'] += f'{col} outlier (lejos de la mediana); '

for c in rasgo_cols:
    marca_outliers(c)

print('Filas marcadas:', (df['QC_flag'] != '').sum(), 'de', len(df))

## 5. Ver y guardar el reporte de filas marcadas

In [ ]:
reporte = df[df['QC_flag'] != ''][
    ['plot ID', 'treeID', 'new tree ID 2025', 'genus', 'species', 'QC_flag']
]
display(reporte)

with pd.ExcelWriter('Rasgos_2025_revisado.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Rasgos_2025_QC', index=False)
    reporte.to_excel(writer, sheet_name='Filas_a_revisar', index=False)

print('Guardado: Rasgos_2025_revisado.xlsx')